# Dynamic Spectrum Access Final Experiment Notebook

This notebook runs the main DSA experiments used for the project submission. It trains/evaluates baseline, ANN, recurrent ANN, STDP SNN, and recurrent SNN policies, writes CSV metrics, saves plots, and displays those plots inline.

**Runtime note:** the full SNN experiment can take a long time. The default settings below are intentionally the final-report style settings, not a smoke test.

## Imports

In [ ]:
import csv
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np

from run_experiments import MODEL_LABELS, run_model, summarize, write_csv

## Experiment Configuration

The default configuration runs the main fixed-environment experiment with 100k steps, three learner seeds, and the models that make sense for the final comparison.

- `env_seed=0` fixes the DSA environment.
- `seeds=[0,1,2]` varies learner initialization/stochasticity.
- `freeze_snn_on_converged=True` prevents late-stage SNN plasticity drift.

To make a faster local check, reduce `steps`, reduce `seeds`, or remove the SNN models from `models`.

In [ ]:
out_dir = Path("result/notebook_final_experiment")
out_dir.mkdir(parents=True, exist_ok=True)

models = [
    "safe_sensed",
    "myopic_noisy",
    "actor_critic",
    "recurrent_actor_critic",
    "rstdp_snn",
    "lsm_snn",
]

seeds = [0, 1, 2]

args = SimpleNamespace(
    steps=100000,
    block_size=2000,
    channels=6,
    sus=2,
    sense_error=0.2,
    pu_penalty=-6.0,
    learning_rate=0.01,
    dqn_learn_period=2000,
    dqn_memory_size=2000,
    dqn_replace_target_iter=1,
    freeze_snn_on_converged=True,
    freeze_success_rate=0.96,
    freeze_pu_collision_rate=0.035,
    freeze_su_collision_rate=0.005,
    env_seed=0,
)

## Run Experiments

This cell writes `block_metrics.partial.csv` after each block, so progress is preserved even if the notebook is interrupted.

In [ ]:
all_rows = []
partial_path = out_dir / "block_metrics.partial.csv"

for seed in seeds:
    for model in models:
        label = MODEL_LABELS.get(model, model)
        print(f"Running {label}, learner seed={seed}, env seed={args.env_seed}")
        current_rows = []

        def record_block(row):
            current_rows.append(dict(row))
            write_csv(all_rows + current_rows, partial_path)
            print(
                f"  block={row['block']:>2} step={row['step']:>6} "
                f"reward={row['avg_reward']:.3f} "
                f"PU/access={row['pu_collision_rate_per_access']:.3f} "
                f"SU/access={row['su_collision_rate_per_access']:.3f} "
                f"frozen={row['learning_frozen']}"
            )

        run_model(model, args, seed, block_callback=record_block)
        all_rows.extend(current_rows)

write_csv(all_rows, out_dir / "block_metrics.csv")
summary_rows = summarize(all_rows)
write_csv(summary_rows, out_dir / "summary_final_block.csv")

summary_rows

## Final Summary Table

In [ ]:
for row in summary_rows:
    print(
        f"{row['model']}: "
        f"reward={row['avg_reward']:.3f}, "
        f"success/access={row['success_rate_per_access']:.3f}, "
        f"PU/access={row['pu_collision_rate_per_access']:.3f}, "
        f"SU/access={row['su_collision_rate_per_access']:.3f}, "
        f"access={row['access_rate']:.3f}, "
        f"spikes/SU-step={row['avg_spikes_per_su_step']:.1f}"
    )

## Plot Helpers

The plots are saved to disk and displayed inline.

In [ ]:
REPORT_LABELS = {
    "SafeSensed": "Safe Sensed",
    "NoisyMyopic": "Noisy Myopic",
    "OnlineActorCritic": "ANN",
    "RecurrentActorCritic": "Recurrent ANN",
    "OnlineActorCriticRSTDP": "STDP SNN",
    "OnlineActorCriticLSM": "Recurrent SNN",
}

def plot_training_metric(rows, metric, ylabel, filename, include_baselines=True):
    fig, ax = plt.subplots(figsize=(12, 7))
    models_in_rows = sorted({row["model"] for row in rows})
    for model_name in models_in_rows:
        if not include_baselines and model_name in {"SafeSensed", "NoisyMyopic", "OracleMyopic", "Myopic"}:
            continue
        model_rows = [row for row in rows if row["model"] == model_name]
        steps = sorted({int(row["step"]) for row in model_rows})
        means = []
        stds = []
        for step in steps:
            vals = [float(row[metric]) for row in model_rows if int(row["step"]) == step]
            means.append(float(np.mean(vals)))
            stds.append(float(np.std(vals)))
        means = np.array(means)
        stds = np.array(stds)
        label = REPORT_LABELS.get(model_name, model_name)
        ax.plot(steps, means, linewidth=3, label=label)
        if len(model_rows) > len(steps):
            ax.fill_between(steps, means - stds, means + stds, alpha=0.15)
    ax.set_xlabel("Training step", fontsize=16)
    ax.set_ylabel(ylabel, fontsize=16)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    fig.tight_layout()
    path = out_dir / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {path}")

## Training Curves With Baselines

In [ ]:
plot_training_metric(all_rows, "avg_reward", "Average reward per SU per step", "training_reward_all_models.png", include_baselines=True)
plot_training_metric(all_rows, "pu_collision_rate_per_access", "PU collision rate per access", "training_pu_collision_all_models.png", include_baselines=True)
plot_training_metric(all_rows, "su_collision_rate_per_access", "SU collision rate per access", "training_su_collision_all_models.png", include_baselines=True)
plot_training_metric(all_rows, "success_rate_per_access", "Success rate per access", "training_success_all_models.png", include_baselines=True)

## Training Curves Without Simple Baselines

These are often easier to read in a report because the simple baselines have very different SU-collision behavior.

In [ ]:
plot_training_metric(all_rows, "avg_reward", "Average reward per SU per step", "training_reward_learning_models.png", include_baselines=False)
plot_training_metric(all_rows, "pu_collision_rate_per_access", "PU collision rate per access", "training_pu_collision_learning_models.png", include_baselines=False)
plot_training_metric(all_rows, "su_collision_rate_per_access", "SU collision rate per access", "training_su_collision_learning_models.png", include_baselines=False)
plot_training_metric(all_rows, "avg_spikes_per_su_step", "Average spikes per SU step", "training_spikes_learning_models.png", include_baselines=False)

## Output Files

In [ ]:
print(f"Experiment outputs saved in: {out_dir}")
for path in sorted(out_dir.iterdir()):
    print(path.name)